In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import shutil

# Set your destination path in Google Drive
destination_path = "/content/drive/My Drive/My-Results_Without_Scale"

# Create the folder if it doesn't exist
import os
os.makedirs(destination_path, exist_ok=True)



## PLSR

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("Complete-WihtotuSclae.csv")
df = df.drop(columns=["Sample"], errors="ignore")

X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values

import numpy as np


print("Any NaNs in X:", np.isnan(X_raw).any())
print("NaNs per column:", np.isnan(X_raw).sum())
print("NaNs per row:", np.isnan(X_raw).sum(axis=1))
nan_cols = df.drop(columns=["N%"]).isna().sum()
nan_cols[nan_cols > 0]


# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
savgol_params_short = {"window_length": 5, "polyorder": 2}  # Optional for savgol only

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),

    # Combined with MSC
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {'n_components': [1,2,3,4,5,6,7,8,9, 10,11,12,13,14,15,16,17,18,19,20]}
repeats = 10
summary_data = {}

# === Run Comparison ===
output_filename = "PLSR_Full_Preprocessing_Comparison.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🔁 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)
            grid = GridSearchCV(PLSRegression(), param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best n_components": grid.best_params_["n_components"]
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best n_components"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best n_components"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All preprocessing comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ PLSR File moved to Google Drive at {destination_path}/{output_filename}")

Any NaNs in X: True
NaNs per column: 140
NaNs per row: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1]
Unique NaN columns: [202]
🔁 Running: Raw


ValueError: 
All the 100 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py", line 719, in fit
    super().fit(X, y)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py", line 265, in fit
    X = validate_data(
        ^^^^^^^^^^^^^^
  File "/usr/loc

# **SVR**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("My_Data_Npercent_Outliers_Removed.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values


# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100, 1000],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}
summary_data = {}

# === Run Comparison ===
output_filename = "SVR_Full_Preprocessing_Comparison.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🔁 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)
            grid = GridSearchCV(SVR(), param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All SVR comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ SVR File moved to Google Drive at {destination_path}/{output_filename}")


🔁 Running: Raw
🔁 Running: MSC
🔁 Running: SNV
🔁 Running: Savitzky-Golay
🔁 Running: First Derivative
🔁 Running: Second Derivative
🔁 Running: Log First Derivative
🔁 Running: Log Second Derivative
🔁 Running: MSC + First Derivative
🔁 Running: MSC + Second Derivative
🔁 Running: MSC + Savitzky-Golay
🔁 Running: MSC + Log First Derivative
🔁 Running: MSC + Log Second Derivative
✅ All SVR comparisons saved to SVR_Full_Preprocessing_Comparison.xlsx
✅ SVR File moved to Google Drive at /content/drive/My Drive/My-Results10122025/SVR_Full_Preprocessing_Comparison.xlsx


# XGBoost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.signal import savgol_filter
from xgboost import XGBRegressor
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("My_Data_Npercent_Outliers_Removed.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values


# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {
    'n_estimators': [100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
}
summary_data = {}
output_filename = "XGBoost_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🚀 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = XGBRegressor(objective='reg:squarederror', verbosity=0)
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All XGBoost comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ XGBoost File moved to Google Drive at {destination_path}/{output_filename}")



🚀 Running: Raw
🚀 Running: MSC
🚀 Running: SNV
🚀 Running: Savitzky-Golay
🚀 Running: First Derivative
🚀 Running: Second Derivative
🚀 Running: Log First Derivative
🚀 Running: Log Second Derivative
🚀 Running: MSC + First Derivative
🚀 Running: MSC + Second Derivative
🚀 Running: MSC + Savitzky-Golay
🚀 Running: MSC + Log First Derivative
🚀 Running: MSC + Log Second Derivative
✅ All XGBoost comparisons saved to XGBoost_Full_Preprocessing_Comparison.xlsx
✅ XGBoost File moved to Google Drive at /content/drive/My Drive/My-Results10122025/XGBoost_Full_Preprocessing_Comparison.xlsx


# **RFR**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("My_Data_Npercent_Outliers_Removed.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values


# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "max_features": ['sqrt', 'log2']
}
summary_data = {}
output_filename = "RandomForest_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🌲 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = RandomForestRegressor(random_state=seed)
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All Random Forest comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ Random Forest File moved to Google Drive at {destination_path}/{output_filename}")

🌲 Running: Raw
🌲 Running: MSC
🌲 Running: SNV
🌲 Running: Savitzky-Golay
🌲 Running: First Derivative
🌲 Running: Second Derivative
🌲 Running: Log First Derivative
🌲 Running: Log Second Derivative
🌲 Running: MSC + First Derivative
🌲 Running: MSC + Second Derivative
🌲 Running: MSC + Savitzky-Golay
🌲 Running: MSC + Log First Derivative
🌲 Running: MSC + Log Second Derivative
✅ All Random Forest comparisons saved to RandomForest_Full_Preprocessing_Comparison.xlsx
✅ Random Forest File moved to Google Drive at /content/drive/My Drive/My-Results10122025/RandomForest_Full_Preprocessing_Comparison.xlsx


# **Ridge Regression**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("My_Data_Npercent_Outliers_Removed.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values

# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Ridge Regression Config ===
param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}
summary_data = {}
output_filename = "RidgeRegression_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🔵 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = Ridge()
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All Ridge Regression comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ Ridge Regression File moved to Google Drive at {destination_path}/{output_filename}")

🔵 Running: Raw
🔵 Running: MSC
🔵 Running: SNV
🔵 Running: Savitzky-Golay
🔵 Running: First Derivative
🔵 Running: Second Derivative
🔵 Running: Log First Derivative
🔵 Running: Log Second Derivative
🔵 Running: MSC + First Derivative
🔵 Running: MSC + Second Derivative
🔵 Running: MSC + Savitzky-Golay
🔵 Running: MSC + Log First Derivative
🔵 Running: MSC + Log Second Derivative
✅ All Ridge Regression comparisons saved to RidgeRegression_Full_Preprocessing_Comparison.xlsx
✅ Ridge Regression File moved to Google Drive at /content/drive/My Drive/My-Results10122025/RidgeRegression_Full_Preprocessing_Comparison.xlsx


# **Decision Tree**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("My_Data_Npercent_Outliers_Removed.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values


# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
summary_data = {}
output_filename = "DecisionTree_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🌳 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = DecisionTreeRegressor(random_state=seed)
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All Decision Tree comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ All Decision File moved to Google Drive at {destination_path}/{output_filename}")

🌳 Running: Raw
🌳 Running: MSC
🌳 Running: SNV
🌳 Running: Savitzky-Golay
🌳 Running: First Derivative
🌳 Running: Second Derivative
🌳 Running: Log First Derivative
🌳 Running: Log Second Derivative
🌳 Running: MSC + First Derivative
🌳 Running: MSC + Second Derivative
🌳 Running: MSC + Savitzky-Golay
🌳 Running: MSC + Log First Derivative
🌳 Running: MSC + Log Second Derivative
✅ All Decision Tree comparisons saved to DecisionTree_Full_Preprocessing_Comparison.xlsx


**LASSO**

In [ ]:
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Lasso, LinearRegression
from scipy.signal import savgol_filter

# ===================== I/O config =====================
DATA_CSV       = "Complete-WithoutScale-CLEAN.csv"
OUTPUT_XLSX    = "LassoRegression_Full_Preprocessing_Comparison.xlsx"
MODELS_DIR     = "SavedModels"  # where best models will be saved
os.makedirs(MODELS_DIR, exist_ok=True)



# ===================== Load data =====================
df = pd.read_csv(DATA_CSV)
df = df.drop(columns=["Sample"], errors="ignore")

feature_cols = [c for c in df.columns if c != "N%"]
X_raw = df[feature_cols].to_numpy(dtype=float)
y     = df["N%"].to_numpy(dtype=float)

# ===================== Preprocessing =====================
def apply_msc(X):
    mean = np.mean(X, axis=0)
    out = np.empty_like(X, dtype=float)
    for i, x in enumerate(X):
        slope, intercept = np.polyfit(mean, x, 1)
        out[i] = (x - intercept) / (slope + 1e-12)
    return out

def apply_snv(X):
    mu = np.mean(X, axis=1, keepdims=True)
    sd = np.std(X, axis=1, keepdims=True)
    return (X - mu) / (sd + 1e-12)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'],
                         deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'],
                         deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'],
                         deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(np.clip(X, 0, None))
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(np.clip(X, 0, None))
    return apply_second_derivative(X, params)

savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# ===================== Lasso config =====================
param_grid = {"alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]}

# ===================== Helpers =====================
rmse = lambda yt, yp: float(np.sqrt(mean_squared_error(yt, yp)))
mae  = lambda yt, yp: float(mean_absolute_error(yt, yp))

def score_tuple(d):
    """Bigger is better. Primary: Test R². Tiebreaker: lower Test RMSE."""
    return (d["Test R²"], -d["Test RMSE"])

def sanitize_filename(s: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in s)

# ===================== Run comparison + save best per method =====================
summary_data = {}
leaderboard_rows = []   # one row per method, with best metrics among repeats

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🟠 Running: {method_name}")
        results = []
        best_run = None   # dict with metrics + objects to save

        X_processed = preprocess_func(X_raw).astype(float)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(
                X_processed, y, test_size=0.1, random_state=seed
            )

            # Lasso CV on training set
            grid = GridSearchCV(
                Lasso(max_iter=10000), param_grid,
                cv=5, scoring="neg_mean_squared_error", n_jobs=-1
            )
            grid.fit(X_train, y_train)
            best_lasso = grid.best_estimator_

            # Lasso-selected indices
            selected_indices = np.where(best_lasso.coef_ != 0)[0]

            # If nothing selected, fallback to constant predictor
            if selected_indices.size == 0:
                y_train_pred = np.full_like(y_train, y_train.mean(), dtype=float)
                y_val_pred   = np.full_like(y_train, y_train.mean(), dtype=float)
                y_test_pred  = np.full_like(y_test,  y_train.mean(), dtype=float)
                ols = None
            else:
                Xtr_sel = X_train[:, selected_indices]
                Xte_sel = X_test[:, selected_indices]

                # OLS refit on selected features
                ols = LinearRegression().fit(Xtr_sel, y_train)
                y_train_pred = ols.predict(Xtr_sel)
                y_val_pred   = cross_val_predict(ols, Xtr_sel, y_train, cv=5)
                y_test_pred  = ols.predict(Xte_sel)

            row = {
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": rmse(y_train, y_train_pred),
                "Train MAE": mae(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": rmse(y_train, y_val_pred),
                "Val MAE": mae(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": rmse(y_test, y_test_pred),
                "Test MAE": mae(y_test, y_test_pred),
                "Test RPD": (float(np.std(y_test)) / rmse(y_test, y_test_pred)) if rmse(y_test, y_test_pred) > 0 else np.nan,
                "Best Params": str(grid.best_params_),
                "Selected Features": int(selected_indices.size),
            }
            results.append(row)

            # Track the best run for this method
            candidate = {
                **row,
                "ols": ols,
                "selected_indices": selected_indices,
                "best_alpha": grid.best_params_.get("alpha"),
            }
            if (best_run is None) or (score_tuple(candidate) > score_tuple(best_run)):
                best_run = candidate

        # Write per-method sheet
        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean(numeric_only=True)
        df_result.loc["Std"]  = df_result.drop(columns=["Repeat", "Best Params"]).std(numeric_only=True)
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)

        # For summary sheet
        summary_data[method_name] = (
            df_result.loc["Mean"].round(6).astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)
        )

        # ====== SAVE the best model for this preprocessing method ======
        safe_name = sanitize_filename(method_name)
        bundle_path = os.path.join(MODELS_DIR, f"{safe_name}_BEST.joblib")
        meta_path   = os.path.join(MODELS_DIR, f"{safe_name}_BEST_meta.json")

        # Build savable bundle (OLS refit + indices + alpha + column names)
        bundle = {
            "preprocessing": method_name,
            "feature_cols": feature_cols,                          # original column names
            "selected_indices": best_run["selected_indices"],      # numpy array
            "best_alpha": best_run["best_alpha"],
            "ols": best_run["ols"],                                # LinearRegression or None
        }
        joblib.dump(bundle, bundle_path)

        with open(meta_path, "w") as f:
            json.dump({
                "method": method_name,
                "best_alpha": best_run["best_alpha"],
                "selected_features": int(best_run["Selected Features"]),
                "metrics": {
                    "train_r2":  best_run["Train R²"],
                    "train_rmse": best_run["Train RMSE"],
                    "train_mae":  best_run["Train MAE"],
                    "val_r2":    best_run["Val R²"],
                    "val_rmse":  best_run["Val RMSE"],
                    "val_mae":   best_run["Val MAE"],
                    "test_r2":   best_run["Test R²"],
                    "test_rmse": best_run["Test RMSE"],
                    "test_mae":  best_run["Test MAE"],
                    "test_rpd":  best_run["Test RPD"],
                },
                "repeat_id": int(best_run["Repeat"]),
                "model_bundle_path": bundle_path
            }, f, indent=2)

        # Add to leaderboard
        leaderboard_rows.append({
            "Method": method_name,
            "Repeat": int(best_run["Repeat"]),
            "Best alpha": best_run["best_alpha"],
            "Selected features": int(best_run["Selected Features"]),
            "Test R²": best_run["Test R²"],
            "Test RMSE": best_run["Test RMSE"],
            "Test MAE": best_run["Test MAE"],
            "Test RPD": best_run["Test RPD"],
            "Bundle": bundle_path
        })

# Write summary sheet
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary", index=True)

# ===================== Leaderboard & display =====================
leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    by=["Test R²", "Test RMSE"], ascending=[False, True]
).reset_index(drop=True)

leaderboard_path = os.path.join(MODELS_DIR, "leaderboard.csv")
leaderboard.to_csv(leaderboard_path, index=False)

# ---- Print top-5 and overall best
print("\n🏆 Top 5 methods by Test R² (tie-break: lower Test RMSE):")
print(leaderboard.head(5).to_string(index=False))

best_row = leaderboard.iloc[0]
print("\n✅ Overall BEST:")
print(
    f"Method: {best_row['Method']} | Repeat: {best_row['Repeat']} | "
    f"Alpha: {best_row['Best alpha']} | Selected: {best_row['Selected features']} | "
    f"Test R²: {best_row['Test R²']:.3f} | RMSE: {best_row['Test RMSE']:.3f} | "
    f"MAE: {best_row['Test MAE']:.3f} | RPD: {best_row['Test RPD']:.3f}\n"
    f"Saved bundle: {best_row['Bundle']}"
)

print(f"\n📁 All best-per-method bundles + metadata are in: {MODELS_DIR}")
print(f"📄 Leaderboard CSV saved to: {leaderboard_path}")
print(f"📘 Full Excel with all repeats: {OUTPUT_XLSX}")

print(f"✅ All Decision Tree comparisons saved to {output_filename}")
shutil.move(output_filename, f"{destination_path}/{output_filename}")
print(f"✅ All Decision File moved to Google Drive at {destination_path}/{output_filename}")

🟠 Running: Raw
🟠 Running: MSC
🟠 Running: SNV
🟠 Running: Savitzky-Golay
🟠 Running: First Derivative
🟠 Running: Second Derivative
🟠 Running: Log First Derivative
🟠 Running: Log Second Derivative
🟠 Running: MSC + First Derivative
🟠 Running: MSC + Second Derivative
🟠 Running: MSC + Savitzky-Golay
🟠 Running: MSC + Log First Derivative
🟠 Running: MSC + Log Second Derivative

🏆 Top 5 methods by Test R² (tie-break: lower Test RMSE):
              Method  Repeat  Best alpha  Selected features  Test R²  Test RMSE  Test MAE  Test RPD                                       Bundle
MSC + Savitzky-Golay       1      0.0001                 17 0.935464   0.176872  0.148391  3.936410 SavedModels/MSC___Savitzky-Golay_BEST.joblib
                 MSC       1      0.0001                 16 0.929272   0.185164  0.153421  3.760135                  SavedModels/MSC_BEST.joblib
                 Raw       5      0.0001                 24 0.926703   0.258385  0.194246  3.693658                  SavedModels/Raw_BES

NameError: name 'output_filename' is not defined

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ================= CONFIG =================
MODEL_PATH = "./Raw_BEST.joblib"
CSV_PATH   = "1.csv"
Y_TRUE_COL = "N%"
OUT_DIR    = "Raw_Model_Prediction_Results"
os.makedirs(OUT_DIR, exist_ok=True)

# ============== Load Model ==============
bundle = joblib.load(MODEL_PATH)
ols = bundle["ols"]
sel = np.array(bundle["selected_indices"])
feature_cols = np.array(bundle["feature_cols"])

print(f"✅ Loaded model ({bundle['preprocessing']}) with {len(sel)} selected features")

# ============== Load Data ==============
df = pd.read_csv(CSV_PATH)

# --- Drop any non-numeric columns (e.g., Name, ID, etc.) ---
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if Y_TRUE_COL not in numeric_cols:
    numeric_cols.append(Y_TRUE_COL)
df = df[numeric_cols].copy()

# --- Extract numeric wavelength values ---
def extract_numeric_wavelength(s):
    s = str(s).replace("λ", "").replace("Λ", "").replace("lam", "")
    s = s.replace(",", ".").replace(" ", "")
    try:
        return float(s)
    except ValueError:
        return None

model_waves = np.array([extract_numeric_wavelength(c) for c in feature_cols if extract_numeric_wavelength(c) is not None])
csv_waves   = np.array([extract_numeric_wavelength(c) for c in df.columns if extract_numeric_wavelength(c) is not None])

# --- Match wavelengths numerically ---
tolerance = 0.05  # nm tolerance
matched_indices = []
for mw in model_waves[sel]:
    diffs = np.abs(csv_waves - mw)
    if len(diffs) > 0 and np.min(diffs) <= tolerance:
        matched_indices.append(np.argmin(diffs))

print(f"⚙️ Matched {len(matched_indices)} / {len(sel)} wavelengths within ±{tolerance} nm")

# --- Select features safely ---
if not matched_indices:
    raise ValueError("No matching wavelengths found — check column headers or tolerance range.")

X = df.iloc[:, [i for i in matched_indices if i < df.shape[1]]].to_numpy(dtype=float)
y_true = pd.to_numeric(df[Y_TRUE_COL], errors="coerce").to_numpy(dtype=float)

print(f"✅ Using {X.shape[1]} features for prediction")

# ============== Predict ==============
y_pred = ols.predict(X)
df["Predicted_N%"] = y_pred

# ============== Metrics ==============
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
rpd = np.std(y_true) / rmse

print("\n📊 Evaluation Results:")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RPD:  {rpd:.4f}")

# ============== Plot ==============
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, c='blue', alpha=0.7)
m, M = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
plt.plot([m, M], [m, M], 'r--')
plt.xlabel("Measured N%")
plt.ylabel("Predicted N%")
plt.title(f"Raw Model Parity Plot (R²={r2:.3f}, RMSE={rmse:.3f})")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "parity_plot.png"), dpi=300)
plt.close()

# ============== Save Results ==============
out_csv = os.path.join(OUT_DIR, "predictions.csv")
df.to_csv(out_csv, index=False)

print(f"\n✅ Predictions saved to: {out_csv}")
print(f"📈 Parity plot saved to: {os.path.join(OUT_DIR, 'parity_plot.png')}")


✅ Loaded model (Raw) with 21 selected features
⚙️ Matched 21 / 21 wavelengths within ±0.05 nm
✅ Using 21 features for prediction

📊 Evaluation Results:
R²:   -613033.4253
RMSE: 799.2047
MAE:  693.4215
RPD:  0.0013

✅ Predictions saved to: Raw_Model_Prediction_Results/predictions.csv
📈 Parity plot saved to: Raw_Model_Prediction_Results/parity_plot.png


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ================= CONFIG =================
MODEL_PATH = "./Raw_BEST.joblib"
CSV_PATH   = "My_Data_Final_Front- 21_10_2025(Sheet1).csv"
Y_TRUE_COL = "N%"
TOL_NM     = 0.05       # wavelength tolerance (nm); increase if needed (e.g., 0.5 or 2.0)
OUT_DIR    = "Raw_Model_Prediction_Results"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- helpers ----------
def parse_wave(name: str):
    s = str(name).strip().lower().replace("lambda", "λ").replace("lam", "λ")
    if "λ" in s:
        s = s.split("λ", 1)[1]
    s = s.replace("nm", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan

def metrics(y, yhat):
    r2   = r2_score(y, yhat)
    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    mae  = float(mean_absolute_error(y, yhat))
    rpd  = float(np.std(y, ddof=1) / rmse) if rmse > 0 else np.nan
    return r2, rmse, mae, rpd

# ============== Load model bundle ==============
bundle = joblib.load(MODEL_PATH)
ols           = bundle["ols"]
sel           = np.array(bundle["selected_indices"], dtype=int)
feature_cols  = np.array(bundle["feature_cols"], dtype=object)
print(f"✅ Loaded model ({bundle.get('preprocessing', 'Unknown')}) with {len(sel)} selected features")

# Parse training wavelengths for all feature columns, then keep only the selected ones
train_waves_all = np.array([parse_wave(c) for c in feature_cols], dtype=float)
if np.isnan(train_waves_all).any():
    # If the training columns include any non-wavelength names, sanity-check here
    pass
train_waves_sel = train_waves_all[sel]

# ============== Load new CSV ==============
df = pd.read_csv(CSV_PATH)
# Drop obvious non-feature columns but keep Y_TRUE_COL
df = df.drop(columns=["Name", "Sample"], errors="ignore")
if Y_TRUE_COL not in df.columns:
    raise ValueError(f"'{Y_TRUE_COL}' not found in CSV.")

# Build a mapping: for each column in df (except Y col), get its wavelength and original index
wave_info = []
for j, col in enumerate(df.columns):
    if col == Y_TRUE_COL:
        continue
    w = parse_wave(col)
    wave_info.append((j, col, w))  # j = REAL DataFrame column index

# For each selected training wavelength, find the best matching df column index
matched_df_indices = []
unmatched = []
for w_train in train_waves_sel:
    # find the df column with minimum |Δ|
    diffs = [abs(w - w_train) if np.isfinite(w) else np.inf for (_, _, w) in wave_info]
    k = int(np.argmin(diffs))
    best_df_idx, best_name, best_w = wave_info[k]
    if np.isfinite(best_w) and abs(best_w - w_train) <= TOL_NM:
        matched_df_indices.append(best_df_idx)
    else:
        unmatched.append(w_train)

print(f"⚙️ Matched {len(matched_df_indices)} / {len(sel)} wavelengths within ±{TOL_NM} nm")
if unmatched:
    print("⚠️ Unmatched training wavelengths (first 10):", np.array(unmatched[:10]))

# You should use ALL selected features; otherwise predictions won’t match the trained OLS shape
if len(matched_df_indices) != len(sel):
    raise ValueError(
        f"Only matched {len(matched_df_indices)} of {len(sel)} selected wavelengths.\n"
        f"Increase TOL_NM (e.g., 0.5 or 2.0) or fix column headers."
    )

# Ensure rows with valid ground-truth
y_true = pd.to_numeric(df[Y_TRUE_COL], errors="coerce")
row_mask = np.isfinite(y_true)
y_true = y_true[row_mask].to_numpy(dtype=float)

# Build X in the **same order** as the selected wavelengths
X_all = df.iloc[:, matched_df_indices].to_numpy(dtype=float)
X = X_all[row_mask]

print(f"✅ Using {X.shape[1]} features for prediction on {X.shape[0]} rows")

# ============== Predict & evaluate ==============
y_pred = ols.predict(X)

r2, rmse, mae, rpd = metrics(y_true, y_pred)
print("\n📊 Evaluation Results:")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RPD:  {rpd:.4f}")

# ============== Save results ==============
out_df = df.loc[row_mask].copy()
out_df["Predicted_N%"] = y_pred
out_csv = os.path.join(OUT_DIR, "predictions.csv")
out_df.to_csv(out_csv, index=False)
print(f"\n✅ Predictions saved to: {out_csv}")


✅ Loaded model (Raw) with 21 selected features
⚙️ Matched 21 / 21 wavelengths within ±0.05 nm
✅ Using 21 features for prediction on 63 rows

📊 Evaluation Results:
R²:   -32.5669
RMSE: 1.9790
MAE:  1.6695
RPD:  0.1740

✅ Predictions saved to: Raw_Model_Prediction_Results/predictions.csv


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# ================= CONFIG =================
MODEL_PATH = "./Raw_BEST.joblib"
CSV_PATH   = "1.csv"
Y_TRUE_COL = "N%"
TOL_NM     = 0.05       # try 0.5 or 2.0 if headers differ slightly
OUT_DIR    = "Raw_Model_Prediction_Results"
os.makedirs(OUT_DIR, exist_ok=True)

def parse_wave(name: str):
    s = str(name).strip().lower().replace("lambda", "λ").replace("lam", "λ")
    if "λ" in s:
        s = s.split("λ", 1)[1]
    s = s.replace("nm", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan

def metrics(y, yhat):
    r2   = r2_score(y, yhat)
    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    mae  = float(mean_absolute_error(y, yhat))
    rpd  = float(np.std(y, ddof=1) / rmse) if rmse > 0 else np.nan
    return r2, rmse, mae, rpd

# ----- load model bundle
bundle = joblib.load(MODEL_PATH)
ols           = bundle["ols"]
sel           = np.array(bundle["selected_indices"], dtype=int)
feature_cols  = np.array(bundle["feature_cols"], dtype=object)
print(f"✅ Loaded model ({bundle.get('preprocessing','Unknown')}) with {len(sel)} selected features")

# training wavelengths (for the original feature columns), then select only those used by the model
train_waves_all = np.array([parse_wave(c) for c in feature_cols], dtype=float)
train_waves_sel = train_waves_all[sel]

# ----- load new data
df = pd.read_csv(CSV_PATH)
df = df.drop(columns=["Name", "Sample"], errors="ignore")

if Y_TRUE_COL not in df.columns:
    raise ValueError(f"'{Y_TRUE_COL}' not found in CSV columns.")

# numeric y_true with row mask
y_true_full = pd.to_numeric(df[Y_TRUE_COL], errors="coerce")
row_mask = np.isfinite(y_true_full)
if row_mask.sum() == 0:
    raise ValueError("No valid numeric values in N% after coercion.")
y_true = y_true_full[row_mask].to_numpy(dtype=float)

# build wavelength map of real df column indices (excluding the y column)
wave_info = []
for j, col in enumerate(df.columns):
    if col == Y_TRUE_COL:
        continue
    wave_info.append((j, col, parse_wave(col)))  # j is REAL df index

# match each selected training wavelength to a df column within tolerance
matched_df_indices = []
unmatched = []
for w_train in train_waves_sel:
    diffs = [abs(w - w_train) if np.isfinite(w) else np.inf for (_, _, w) in wave_info]
    k = int(np.argmin(diffs))
    j_idx, name_j, w_j = wave_info[k]
    if np.isfinite(w_j) and abs(w_j - w_train) <= TOL_NM:
        matched_df_indices.append(j_idx)
    else:
        unmatched.append(w_train)

print(f"⚙️ Matched {len(matched_df_indices)} / {len(sel)} wavelengths within ±{TOL_NM} nm")
if unmatched:
    print("⚠️ Unmatched (first 10):", np.array(unmatched[:10]))

# must have all selected features for the OLS (21)
if len(matched_df_indices) != len(sel):
    raise ValueError(
        f"Matched {len(matched_df_indices)} of {len(sel)} selected wavelengths. "
        f"Increase TOL_NM or fix column names."
    )

# construct X in the exact trained order and apply the same row_mask as y_true
X_all = df.iloc[:, matched_df_indices].to_numpy(dtype=float)
X = X_all[row_mask]
print(f"✅ Using {X.shape[1]} features for prediction on {X.shape[0]} rows")
assert X.shape[1] == getattr(ols, "n_features_in_", X.shape[1]), "Feature count mismatch to OLS"

# ----- predict & evaluate
y_pred = ols.predict(X)
r2, rmse, mae, rpd = metrics(y_true, y_pred)
print("\n📊 Evaluation Results:")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RPD:  {rpd:.4f}")

# sanity check before plotting
print("y_true min/max:", np.min(y_true), np.max(y_true))
print("y_pred min/max:", np.min(y_pred), np.max(y_pred))

# ----- plot
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.7)
m, M = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))
plt.plot([m, M], [m, M], 'r--')
plt.xlabel("Measured N%")
plt.ylabel("Predicted N%")
plt.title(f"Raw Model Parity Plot (R²={r2:.3f}, RMSE={rmse:.3f})")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "parity_plot.png"), dpi=300)
plt.close()

# ----- save predictions (only rows used)
out_df = df.loc[row_mask].copy()
out_df["Predicted_N%"] = y_pred
out_df.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False)
print(f"\n✅ Saved: {os.path.join(OUT_DIR, 'predictions.csv')}")
print(f"📈 Saved: {os.path.join(OUT_DIR, 'parity_plot.png')}")


✅ Loaded model (Raw) with 21 selected features
⚙️ Matched 21 / 21 wavelengths within ±0.05 nm
✅ Using 21 features for prediction on 137 rows

📊 Evaluation Results:
R²:   0.8574
RMSE: 0.3855
MAE:  0.2882
RPD:  2.6575
y_true min/max: 0.922 6.386
y_pred min/max: 0.6941001486894267 4.9301970486132385

✅ Saved: Raw_Model_Prediction_Results/predictions.csv
📈 Saved: Raw_Model_Prediction_Results/parity_plot.png


In [ ]:
# adapt_model_to_new_data.py
import os, json, joblib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===================== USER CONFIG =====================
MODEL_PATH   = "Raw_BEST.joblib"  # best bundle you saved during training
NEW_CSV_PATH = "1.csv"  # your new file
Y_COL        = "N%"               # truth column name (if present)
TOL_NM       = 2.0                # wavelength matching tolerance in nm
OUT_DIR      = "AdaptedModels"    # where to save results/bundles/report
os.makedirs(OUT_DIR, exist_ok=True)

# =============== small helpers ===============
def metrics(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    r2   = float(r2_score(y, yhat))
    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    mae  = float(mean_absolute_error(y, yhat))
    rpd  = float(np.std(y, ddof=1) / rmse) if rmse > 0 else np.nan
    return dict(R2=r2, RMSE=rmse, MAE=mae, RPD=rpd)

def parse_wavelengths(cols):
    """
    Accepts names like 'λ650.12' or raw floats/strings ('650.12').
    Returns np.ndarray[float] of same length; non-wavelength columns become np.nan.
    """
    out = []
    for c in cols:
        s = str(c).strip().lower().replace("lam", "λ")  # in case column uses 'lam'
        s = s.replace("lambda", "λ")
        if "λ" in s:
            s = s.split("λ", 1)[1]
        # strip trailing units/whitespace
        s = s.replace("nm", "").strip()
        try:
            out.append(float(s))
        except:
            out.append(np.nan)
    return np.array(out, dtype=float)

def wavelength_match(train_waves, new_waves, selected_idx, tol_nm):
    """
    For each selected training index, find the new index whose wavelength is
    within tol_nm and closest in absolute difference.
    Returns: matched_new_idx (list) and mask (bool) of which were matched.
    """
    matched_idx = []
    matched_ok  = []
    for i in selected_idx:
        w = train_waves[i]
        diffs = np.abs(new_waves - w)
        j = int(np.argmin(diffs))
        if np.isfinite(new_waves[j]) and diffs[j] <= tol_nm:
            matched_idx.append(j)
            matched_ok.append(True)
        else:
            matched_idx.append(-1)
            matched_ok.append(False)
    return matched_idx, np.array(matched_ok, dtype=bool)

def ensure_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)

# ===================== LOAD MODEL BUNDLE =====================
print("🔹 Loading model bundle…")
bundle = joblib.load(MODEL_PATH)
method         = bundle.get("preprocessing", "Unknown")
feature_cols   = bundle["feature_cols"]     # training feature column names
selected_idx   = np.array(bundle["selected_indices"], dtype=int)  # indices into feature_cols
base_model     = bundle["ols"]              # LinearRegression refit on selected features
if base_model is None:
    raise RuntimeError("Bundle has no 'ols' model. Re-run training saver to include OLS.")

print(f"✅ Loaded bundle: method={method}, selected_features={len(selected_idx)}")

# Parse training wavelengths (use only wavelength columns; non-wavelength => nan)
train_waves_full = parse_wavelengths(feature_cols)
if np.isnan(train_waves_full).all():
    raise RuntimeError("Training feature columns do not contain wavelengths like 'λ650.12'.")

# ===================== LOAD NEW DATA =====================
df = pd.read_csv(NEW_CSV_PATH)
# drop obvious non-feature name-ish columns if present
df = df.drop(columns=["Name", "Sample"], errors="ignore")

# figure out new file wavelength columns
new_cols = list(df.columns)
if Y_COL in df.columns:
    new_cols_no_y = [c for c in new_cols if c != Y_COL]
else:
    new_cols_no_y = new_cols  # allow prediction-only files

new_waves_full = parse_wavelengths(new_cols_no_y)

# subset to rows where y is available (for evaluation). If Y_COL absent, we'll skip metrics.
has_y = Y_COL in df.columns
if has_y:
    y_all = ensure_numeric(df[Y_COL])
    row_mask = np.isfinite(y_all)
else:
    y_all = None
    row_mask = np.ones(len(df), dtype=bool)

# ===================== MATCH SELECTED FEATURES =====================
matched_new_idx, ok = wavelength_match(train_waves_full, new_waves_full, selected_idx, TOL_NM)
n_ok = int(ok.sum())
print(f"🔧 Selected features in model: {len(selected_idx)}")
print(f"⚙️ Matched {n_ok} / {len(selected_idx)} wavelengths within ±{TOL_NM} nm")

if n_ok == 0:
    raise RuntimeError("No selected wavelengths matched in the new file. Increase TOL_NM or check headers.")

# build X_new (ordered to match the training selected order), only for matched ones
sel_keep_train = selected_idx[ok]
sel_keep_new   = np.array([matched_new_idx[k] for k,good in enumerate(ok) if good], dtype=int)

X_all_full = df[new_cols_no_y].to_numpy(dtype=float)
X_new_all  = X_all_full[:, sel_keep_new]      # (N, n_ok)
rows_used  = np.where(row_mask)[0]
X_new      = X_new_all[row_mask]

print(f"🔹 New data rows usable: {len(rows_used)} | features used: {X_new.shape[1]}")

# if we have fewer than full selected features, we can only do B / C on matched features;
# baseline A requires full feature count for base_model.predict. Handle both:

def try_baseline_predict():
    """Try to run baseline model (needs all selected features). Return (yhat or None, note)."""
    note = ""
    if n_ok < len(selected_idx):
        note = "Baseline skipped (not all selected wavelengths were matched)."
        return None, note
    # full match: reorder to full selected order
    sel_full_new = np.array(matched_new_idx, dtype=int)
    X_all_for_base = X_all_full[:, sel_full_new][row_mask]
    try:
        yhat = base_model.predict(X_all_for_base)
        return yhat, ""
    except Exception as e:
        note = f"Baseline failed: {e}"
        return None, note

# ===================== BASELINE EVALUATION (original model) =====================
report_lines = []
yhat_baseline, base_note = try_baseline_predict()
if has_y and (yhat_baseline is not None):
    m_base = metrics(y_all[row_mask], yhat_baseline)
    report_lines.append(f"📊 Baseline on new data (original model): "
                        f"R²={m_base['R2']:.4f}  RMSE={m_base['RMSE']:.4f}  "
                        f"MAE={m_base['MAE']:.4f}  RPD={m_base['RPD']:.4f}")
    print(report_lines[-1])
else:
    report_lines.append(f"📊 Baseline: {base_note or 'No ground-truth available; metrics skipped.'}")
    print(report_lines[-1])

# ===================== ADAPTATION =====================
adapt_bundles = []

# ---------- A) Bias/scale correction (requires baseline preds) ----------
if yhat_baseline is not None and has_y:
    lr = LinearRegression().fit(yhat_baseline.reshape(-1, 1), y_all[row_mask])
    a, b = float(lr.intercept_), float(lr.coef_[0])
    yhat_A = a + b * yhat_baseline
    mA = metrics(y_all[row_mask], yhat_A)
    print(f"\n🪄 A) Bias/scale: a={a:.6f}, b={b:.6f}  "
          f"R²={mA['R2']:.4f}  RMSE={mA['RMSE']:.4f}  MAE={mA['MAE']:.4f}  RPD={mA['RPD']:.4f}")

    bundle_A = {
        "kind": "bias_scale_on_base",
        "preprocessing": method,
        "feature_names_new": list(np.array(new_cols_no_y)[matched_new_idx]),  # in base order
        "selected_full_new_idx": np.array(matched_new_idx, dtype=int),
        "a": a, "b": b,
        "base_model": base_model,  # original OLS refit
    }
    path_A = os.path.join(OUT_DIR, "Raw_on_NewData_A_bias_scale.joblib")
    joblib.dump(bundle_A, path_A)
    adapt_bundles.append(("A_bias_scale", path_A, mA))
else:
    print("\n🪄 A) Bias/scale: skipped (baseline predictions unavailable).")

# ---------- B) Refit final linear layer (Ridge) on matched features ----------
if has_y:
    ridge = Ridge(alpha=1.0, random_state=0)
    ridge.fit(X_new, y_all[row_mask])
    yhat_B = ridge.predict(X_new)
    mB = metrics(y_all[row_mask], yhat_B)
    print(f"\n🧩 B) Refit final layer (Ridge α=1.0): "
          f"R²={mB['R2']:.4f}  RMSE={mB['RMSE']:.4f}  MAE={mB['MAE']:.4f}  RPD={mB['RPD']:.4f}")

    bundle_B = {
        "kind": "refit_on_new_only",
        "preprocessing": method,
        "feature_names_new": list(np.array(new_cols_no_y)[sel_keep_new]),  # order used for fit
        "model": ridge,
    }
    path_B = os.path.join(OUT_DIR, "Raw_on_NewData_B_refit_new_only.joblib")
    joblib.dump(bundle_B, path_B)
    adapt_bundles.append(("B_refit_new_only", path_B, mB))

# ---------- C) Blended refit with prior (keep close to original coefs) ----------
# Solve: argmin_w ||y - Xw||^2 + λ ||w - w0||^2  -> closed form
# w* = (X^T X + λ I)^(-1) (X^T y + λ w0)
if has_y:
    # original weights for the matched subset.
    # If not full match, slice the original coef_ to the matched features only.
    # We need the *same order* as X_new columns (sel_keep_new).
    # Build a mapping from "kept" training indices -> position within kept set
    # and pull those coefficients.
    if n_ok == len(selected_idx):
        w0 = base_model.coef_.ravel().astype(float)
        b0 = float(base_model.intercept_)
    else:
        # need to reduce to matched subset (same order as sel_keep_train / sel_keep_new)
        full_w = base_model.coef_.ravel().astype(float)
        w0 = full_w[ok]
        b0 = float(base_model.intercept_)  # we keep intercept as a free term

    X = X_new
    y = y_all[row_mask].astype(float)

    lam = 1.0  # strength of "stay close to original weights"; tune if desired
    XtX = X.T @ X
    Xty = X.T @ y
    I   = np.eye(X.shape[1])
    w_blend = np.linalg.solve(XtX + lam * I, Xty + lam * w0)

    # For intercept, fit on residual mean (or do a small linear regression with fixed slope)
    # Here: set intercept to align means
    b_blend = float(y.mean() - X.mean(axis=0) @ w_blend)

    yhat_C = X @ w_blend + b_blend
    mC = metrics(y, yhat_C)
    print(f"\n⚖️  C) Blended refit (λ={lam:.2f}): "
          f"R²={mC['R2']:.4f}  RMSE={mC['RMSE']:.4f}  MAE={mC['MAE']:.4f}  RPD={mC['RPD']:.4f}")

    bundle_C = {
        "kind": "blended_refit_prior",
        "preprocessing": method,
        "feature_names_new": list(np.array(new_cols_no_y)[sel_keep_new]),
        "coef": w_blend,
        "intercept": b_blend,
        "lambda": lam,
        "prior_coef": w0,
        "prior_intercept": b0,
    }
    path_C = os.path.join(OUT_DIR, "Raw_on_NewData_C_blend_refit.joblib")
    joblib.dump(bundle_C, path_C)
    adapt_bundles.append(("C_blend_refit", path_C, mC))

# ===================== SAVE PREDICTIONS + REPORT =====================
# Choose the best adapted prediction (prefer C, else B, else A if available)
best_name, best_path, best_metrics = None, None, None
order = ["C_blend_refit", "B_refit_new_only", "A_bias_scale"]
preds = {}

if has_y:
    if "C_blend_refit" in [n for n,_,_ in adapt_bundles]:
        y_best = yhat_C; best_name, best_path, best_metrics = adapt_bundles[[n for n,_,_ in adapt_bundles].index("C_blend_refit")]
        preds["Adapted"] = yhat_C
    elif "B_refit_new_only" in [n for n,_,_ in adapt_bundles]:
        y_best = yhat_B; best_name, best_path, best_metrics = adapt_bundles[[n for n,_,_ in adapt_bundles].index("B_refit_new_only")]
        preds["Adapted"] = yhat_B
    elif "A_bias_scale" in [n for n,_,_ in adapt_bundles]:
        y_best = yhat_A; best_name, best_path, best_metrics = adapt_bundles[[n for n,_,_ in adapt_bundles].index("A_bias_scale")]
        preds["Adapted"] = yhat_A
    else:
        y_best = None

    # add baseline if we computed it
    if yhat_baseline is not None:
        preds["Baseline"] = yhat_baseline

    # save predictions CSV (only rows with y available)
    out_pred = pd.DataFrame({"Row": rows_used})
    out_pred["Observed_N%"] = y_all[row_mask].values
    for k,v in preds.items():
        out_pred[f"Pred_{k}"] = v
    out_csv = os.path.join(OUT_DIR, "new_data_predictions.csv")
    out_pred.to_csv(out_csv, index=False)
    print(f"\n💾 Saved predictions → {out_csv}")

# human-readable report
if has_y:
    if best_name is not None:
        report_lines.append(f"\n✅ BEST adapted model: {best_name}  "
                            f"(saved: {best_path})\n"
                            f"    R²={best_metrics['R2']:.4f}, RMSE={best_metrics['RMSE']:.4f}, "
                            f"MAE={best_metrics['MAE']:.4f}, RPD={best_metrics['RPD']:.4f}")
    else:
        report_lines.append("\n⚠️ No adapted model selected (check matching/labels).")

report_txt = os.path.join(OUT_DIR, "adaptation_report.txt")
with open(report_txt, "w") as f:
    f.write(f"Model: {MODEL_PATH}\nNew file: {NEW_CSV_PATH}\n")
    f.write(f"Matched wavelengths within ±{TOL_NM} nm: {n_ok}/{len(selected_idx)}\n")
    f.write("\n".join(report_lines) + "\n")
print(f"📝 Report saved → {report_txt}")

print("\nDone.")


🔹 Loading model bundle…
✅ Loaded bundle: method=Raw, selected_features=21
🔧 Selected features in model: 21
⚙️ Matched 21 / 21 wavelengths within ±2.0 nm
🔹 New data rows usable: 137 | features used: 21
📊 Baseline on new data (original model): R²=0.8574  RMSE=0.3855  MAE=0.2882  RPD=2.6575

🪄 A) Bias/scale: a=-0.041915, b=1.019135  R²=0.8578  RMSE=0.3850  MAE=0.2890  RPD=2.6613

🧩 B) Refit final layer (Ridge α=1.0): R²=0.6083  RMSE=0.6389  MAE=0.4669  RPD=1.6036

⚖️  C) Blended refit (λ=1.00): R²=0.8558  RMSE=0.3876  MAE=0.2945  RPD=2.6434

💾 Saved predictions → AdaptedModels/new_data_predictions.csv
📝 Report saved → AdaptedModels/adaptation_report.txt

Done.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Lasso
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("Final_Data_Set_With_Nitorgen_Suzzy.csv")
df = df.drop(columns=["Sample", "Name"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values

# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Lasso Regression Config ===
param_grid = {
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
}
summary_data = {}
output_filename = "LassoRegression_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🟠 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = Lasso(max_iter=10000)
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            from sklearn.linear_model import LinearRegression

            # === Step 1: Get selected features from Lasso
            selected_indices = np.where(best_model.coef_ != 0)[0]
            if len(selected_indices) == 0:
                y_test_pred = np.full_like(y_test, y_test.mean())  # fallback if Lasso selects nothing
            else:
                X_train_selected = X_train[:, selected_indices]
                X_test_selected = X_test[:, selected_indices]

                # === Step 2: Refit using OLS
                ols = LinearRegression().fit(X_train_selected, y_train)
                y_train_pred = ols.predict(X_train_selected)
                y_val_pred = cross_val_predict(ols, X_train_selected, y_train, cv=5)
                y_test_pred = ols.predict(X_test_selected)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_),
                "Selected Features": len(selected_indices)

            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All Lasso Regression comparisons saved to {output_filename}")


🟠 Running: Raw
🟠 Running: MSC
🟠 Running: SNV
🟠 Running: Savitzky-Golay
🟠 Running: First Derivative
🟠 Running: Second Derivative
🟠 Running: Log First Derivative
🟠 Running: Log Second Derivative
🟠 Running: MSC + First Derivative
🟠 Running: MSC + Second Derivative
🟠 Running: MSC + Savitzky-Golay
🟠 Running: MSC + Log First Derivative
🟠 Running: MSC + Log Second Derivative
✅ All Lasso Regression comparisons saved to LassoRegression_Full_Preprocessing_Comparison.xlsx


# **KNN**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("Complete-WithoutScale-CLEAN.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values



# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === Model Config ===
param_grid = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]  # 1 = Manhattan, 2 = Euclidean
}
summary_data = {}
output_filename = "KNN_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"📍 Running: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = KNeighborsRegressor()
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All KNN comparisons saved to {output_filename}")


📍 Running: Raw
📍 Running: MSC
📍 Running: SNV
📍 Running: Savitzky-Golay
📍 Running: First Derivative
📍 Running: Second Derivative
📍 Running: Log First Derivative
📍 Running: Log Second Derivative
📍 Running: MSC + First Derivative
📍 Running: MSC + Second Derivative
📍 Running: MSC + Savitzky-Golay
📍 Running: MSC + Log First Derivative
📍 Running: MSC + Log Second Derivative
✅ All KNN comparisons saved to KNN_Full_Preprocessing_Comparison.xlsx


# **ADABoost**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import AdaBoostRegressor
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore")

# === Load CSV Data ===
df = pd.read_csv("Complete-WithoutScale-CLEAN.csv")
df = df.drop(columns=["Sample"], errors="ignore")
X_raw = df.drop(columns=["N%"]).values
y = df["N%"].values



# === Preprocessing Functions ===
def apply_msc(X):
    mean = np.mean(X, axis=0)
    return np.array([(x - np.polyfit(mean, x, 1)[1]) / np.polyfit(mean, x, 1)[0] for x in X])

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X = np.log1p(X)
    return apply_first_derivative(X, params)

def apply_log_second_derivative(X, params):
    X = np.log1p(X)
    return apply_second_derivative(X, params)

# === Parameters ===
savgol_params = {"window_length": 11, "polyorder": 2}
repeats = 10

# === Preprocessing Variants ===
preprocessing_methods = {
    "Raw": lambda X: X,
    "MSC": apply_msc,
    "SNV": apply_snv,
    "Savitzky-Golay": lambda X: apply_savgol(X, savgol_params),
    "First Derivative": lambda X: apply_first_derivative(X, savgol_params),
    "Second Derivative": lambda X: apply_second_derivative(X, savgol_params),
    "Log First Derivative": lambda X: apply_log_first_derivative(X, savgol_params),
    "Log Second Derivative": lambda X: apply_log_second_derivative(X, savgol_params),
    "MSC + First Derivative": lambda X: apply_first_derivative(apply_msc(X), savgol_params),
    "MSC + Second Derivative": lambda X: apply_second_derivative(apply_msc(X), savgol_params),
    "MSC + Savitzky-Golay": lambda X: apply_savgol(apply_msc(X), savgol_params),
    "MSC + Log First Derivative": lambda X: apply_log_first_derivative(apply_msc(X), savgol_params),
    "MSC + Log Second Derivative": lambda X: apply_log_second_derivative(apply_msc(X), savgol_params),
}

# === AdaBoost Hyperparameters ===
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

summary_data = {}
output_filename = "AdaBoost_Full_Preprocessing_Comparison.xlsx"

# === Run Comparison ===
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"⚡ Running AdaBoost: {method_name}")
        results = []
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.1, random_state=seed)

            model = AdaBoostRegressor(random_state=seed)
            grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train)
            y_val_pred = cross_val_predict(best_model, X_train, y_train, cv=5)
            y_test_pred = best_model.predict(X_test)

            results.append({
                "Repeat": seed + 1,
                "Train R²": r2_score(y_train, y_train_pred),
                "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
                "Train MAE": mean_absolute_error(y_train, y_train_pred),
                "Val R²": r2_score(y_train, y_val_pred),
                "Val RMSE": np.sqrt(mean_squared_error(y_train, y_val_pred)),
                "Val MAE": mean_absolute_error(y_train, y_val_pred),
                "Test R²": r2_score(y_test, y_test_pred),
                "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Test MAE": mean_absolute_error(y_test, y_test_pred),
                "Test RPD": np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred)),
                "Best Params": str(grid.best_params_)
            })

        df_result = pd.DataFrame(results)
        df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean()
        df_result.loc["Std"] = df_result.drop(columns=["Repeat", "Best Params"]).std()
        df_result.to_excel(writer, sheet_name=method_name[:31], index=False)
        summary_data[method_name] = df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)

    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All AdaBoost regression comparisons saved to {output_filename}")


⚡ Running AdaBoost: Raw
⚡ Running AdaBoost: MSC
⚡ Running AdaBoost: SNV
⚡ Running AdaBoost: Savitzky-Golay
⚡ Running AdaBoost: First Derivative
⚡ Running AdaBoost: Second Derivative
⚡ Running AdaBoost: Log First Derivative
⚡ Running AdaBoost: Log Second Derivative
⚡ Running AdaBoost: MSC + First Derivative
⚡ Running AdaBoost: MSC + Second Derivative
⚡ Running AdaBoost: MSC + Savitzky-Golay
⚡ Running AdaBoost: MSC + Log First Derivative
⚡ Running AdaBoost: MSC + Log Second Derivative
✅ All AdaBoost regression comparisons saved to AdaBoost_Full_Preprocessing_Comparison.xlsx


In [ ]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.signal import savgol_filter

warnings.filterwarnings("ignore")

# ========== Paths (edit destination_path if you want to move the Excel) ==========
output_filename = "PLSR_Full_Preprocessing_Comparison.xlsx"
plots_dir = "plots"
waves_dir = "wavelengths"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(waves_dir, exist_ok=True)
destination_path = "."  # <--- set this to your Google Drive mount if needed (e.g., "/content/drive/My Drive")

# ========== Load CSV Data ==========
df = pd.read_csv("Final_Data_Set_With_Nitorgen_Suzzy.csv")
df = df.drop(columns=["Sample","Name"], errors="ignore")

target_col = "N%"
feature_df = df.drop(columns=[target_col])
wavelength_names = feature_df.columns.to_numpy()  # keep names to export later
X_raw = feature_df.values
y = df[target_col].values

# ========== Optional Preprocessing (you can add more later) ==========
def apply_msc(X):
    mean = np.mean(X, axis=0)
    out = []
    for x in X:
        b, a = np.polyfit(mean, x, 1)
        out.append((x - a) / b)
    return np.asarray(out)

def apply_snv(X):
    return (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

def apply_savgol(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=0, axis=1)

def apply_first_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=1, axis=1)

def apply_second_derivative(X, params):
    return savgol_filter(X, window_length=params['window_length'], polyorder=params['polyorder'], deriv=2, axis=1)

def apply_log_first_derivative(X, params):
    X1 = np.log1p(X)
    return apply_first_derivative(X1, params)

def apply_log_second_derivative(X, params):
    X1 = np.log1p(X)
    return apply_second_derivative(X1, params)

savgol_params = {"window_length": 11, "polyorder": 2}
savgol_params_short = {"window_length": 5, "polyorder": 2}

preprocessing_methods = {
    "Raw": lambda X: X,

}

# ========== VIP computation ==========
def compute_vip(pls: PLSRegression, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    VIP scores for PLS1 (single y). For multi-component PLS, accumulates across components.
    """
    # Based on Wold’s VIP formula
    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_.ravel()  # shape (n_components,)
    p, a = W.shape  # p features, a components

    # Sum of squares explained by each component in y-space
    # For single y, SS_y per component ~ (t_k^2 * q_k^2)
    SS = np.sum((T ** 2) * (Q ** 2), axis=0)  # length a
    total_SS = np.sum(SS)

    vip = np.zeros(p)
    for j in range(p):
        weight = np.array([(W[j, k] ** 2) * SS[k] for k in range(a)])
        vip[j] = np.sqrt(p * np.sum(weight) / (total_SS + 1e-12))
    return vip

# ========== Search grids ==========
param_grid_ncomp = {'n_components': list(range(1, 21))}
# Try these top-k wavelength counts (adjust for your instrument dimensionality)
k_grid = [50, 75, 100, 150, 200, 300, 400, None]  # None => use all wavelengths

repeats = 10
summary_data = {}

# ========== Plot helper ==========
def save_actual_vs_pred_plots(y_true_train, y_pred_train,
                              y_true_val, y_pred_val,
                              y_true_test, y_pred_test,
                              title_prefix: str,
                              outfile: str):
    plt.figure(figsize=(6, 6))
    # Train
    plt.scatter(y_true_train, y_pred_train, alpha=0.5, label=f"Train (R²={r2_score(y_true_train, y_pred_train):.3f})")
    # Val
    plt.scatter(y_true_val, y_pred_val, alpha=0.5, label=f"Val (R²={r2_score(y_true_val, y_pred_val):.3f})")
    # Test
    plt.scatter(y_true_test, y_pred_test, alpha=0.8, label=f"Test (R²={r2_score(y_true_test, y_pred_test):.3f})")

    # 1:1 line
    all_true = np.concatenate([y_true_train, y_true_val, y_true_test])
    lo, hi = np.min(all_true), np.max(all_true)
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)

    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title_prefix)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

# ========== Run ==========
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    for method_name, preprocess_func in preprocessing_methods.items():
        print(f"🔁 Running: {method_name}")
        results = []

        # Preprocess entire X once per method
        X_processed = preprocess_func(X_raw)

        for seed in range(repeats):
            # Split
            X_train, X_test, y_train, y_test = train_test_split(
                X_processed, y, test_size=0.1, random_state=seed
            )

            # --- Stage 1: compute VIP on training using a moderate n_components (auto-bounded) ---
            # Use up to min(15, n_features-1) components for VIP seed model
            base_ncomp = int(min(15, X_train.shape[1] - 1, max(2, np.linalg.matrix_rank(X_train))))
            base_ncomp = max(2, base_ncomp)
            pls_for_vip = PLSRegression(n_components=base_ncomp, scale=False)
            pls_for_vip.fit(X_train, y_train)
            vip_scores = compute_vip(pls_for_vip, X_train, y_train)
            vip_order = np.argsort(vip_scores)[::-1]  # descending

            # --- Stage 2: search over k (top-k wavelengths) and n_components using CV on training ---
            best_cv_mse = np.inf
            best_k = None
            best_n = None
            best_grid = None
            best_sel_idx = None

            for k in k_grid:
                if (k is None) or (k >= X_train.shape[1]):
                    sel_idx = np.arange(X_train.shape[1])
                else:
                    sel_idx = vip_order[:k]

                Xtr_sel = X_train[:, sel_idx]
                # n_components cannot exceed min(n_samples-1, n_features_sel)
                max_comp_allowed = int(min(20, Xtr_sel.shape[0] - 1, Xtr_sel.shape[1]))
                if max_comp_allowed < 1:
                    continue
                grid = GridSearchCV(
                    PLSRegression(scale=False),
                    {'n_components': list(range(1, max_comp_allowed + 1))},
                    cv=5,
                    scoring='neg_mean_squared_error',
                    n_jobs=-1,
                    return_train_score=True
                )
                grid.fit(Xtr_sel, y_train)
                cv_mse = -grid.best_score_
                if cv_mse < best_cv_mse:
                    best_cv_mse = cv_mse
                    best_k = (None if k is None else int(k))
                    best_n = int(grid.best_params_['n_components'])
                    best_grid = grid
                    best_sel_idx = sel_idx

            # --- Stage 3: refit best model on selected wavelengths and evaluate ---
            Xtr_sel = X_train[:, best_sel_idx]
            Xte_sel = X_test[:, best_sel_idx]

            best_model = PLSRegression(n_components=best_n, scale=False)
            best_model.fit(Xtr_sel, y_train)

            y_train_pred = best_model.predict(Xtr_sel).ravel()
            # Cross-validated predictions for validation estimate on TRAIN splits:
            y_val_pred = cross_val_predict(best_model, Xtr_sel, y_train, cv=5).ravel()
            y_test_pred = best_model.predict(Xte_sel).ravel()

            # Metrics
            train_r2 = r2_score(y_train, y_train_pred)
            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            train_mae = mean_absolute_error(y_train, y_train_pred)

            val_r2 = r2_score(y_train, y_val_pred)
            val_rmse = np.sqrt(mean_squared_error(y_train, y_val_pred))
            val_mae = mean_absolute_error(y_train, y_val_pred)

            test_r2 = r2_score(y_test, y_test_pred)
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            test_mae = mean_absolute_error(y_test, y_test_pred)
            test_rpd = np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred))

            # Save plot
            title = f"{method_name} | Repeat {seed+1} | k={best_k if best_k is not None else 'ALL'} | n={best_n}"
            plot_path = os.path.join(plots_dir, f"{method_name}_repeat{seed+1}_k{best_k if best_k is not None else 'ALL'}_n{best_n}.png")
            save_actual_vs_pred_plots(
                y_train, y_train_pred,
                y_train, y_val_pred,
                y_test, y_test_pred,
                title_prefix=title,
                outfile=plot_path
            )

            # Save selected wavelength names for this repeat
            sel_wave_names = wavelength_names[best_sel_idx]
            waves_csv_path = os.path.join(waves_dir, f"{method_name}_repeat{seed+1}_selected_wavelengths.csv")
            pd.Series(sel_wave_names, name="wavelength").to_csv(waves_csv_path, index=False)

            # Append row
            results.append({
                "Repeat": seed + 1,
                "Train R²": train_r2,
                "Train RMSE": train_rmse,
                "Train MAE": train_mae,
                "Val R²": val_r2,
                "Val RMSE": val_rmse,
                "Val MAE": val_mae,
                "Test R²": test_r2,
                "Test RMSE": test_rmse,
                "Test MAE": test_mae,
                "Test RPD": test_rpd,
                "Best n_components": best_n,
                "Best k (wavelengths)": best_k if best_k is not None else len(best_sel_idx),
                "Plot Path": plot_path,
                "Wavelengths CSV": waves_csv_path
            })

        # Per-method sheet
        df_result = pd.DataFrame(results)
        mean_row = df_result.drop(columns=["Repeat", "Best n_components", "Best k (wavelengths)", "Plot Path", "Wavelengths CSV"]).mean(numeric_only=True)
        std_row  = df_result.drop(columns=["Repeat", "Best n_components", "Best k (wavelengths)", "Plot Path", "Wavelengths CSV"]).std(numeric_only=True)
        df_result.loc["Mean"] = pd.concat([mean_row, pd.Series(dtype=float)])
        df_result.loc["Std"]  = pd.concat([std_row,  pd.Series(dtype=float)])

        sheet_name = method_name[:31]
        df_result.to_excel(writer, sheet_name=sheet_name, index=False)

        # Compact summary for the "Summary" sheet (with ±)
        show_cols = ["Train R²", "Val R²", "Test R²", "Train RMSE", "Val RMSE", "Test RMSE", "Test RPD"]
        summary_series = []
        for c in show_cols:
            mu = df_result.loc["Mean", c]
            sd = df_result.loc["Std", c]
            summary_series.append(f"{mu:.4f} ± {sd:.4f}")
        summary_data[method_name] = dict(zip(show_cols, summary_series))

    # Summary sheet
    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

print(f"✅ All results saved to {output_filename}")



🔁 Running: Raw
✅ All results saved to PLSR_Full_Preprocessing_Comparison.xlsx


In [ ]:
import os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# === Paths ===
output_filename = "PLSR_Full_Preprocessing_Comparison2.xlsx"
plots_dir = "plots2"
waves_dir = "wavelengths2"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(waves_dir, exist_ok=True)
destination_path = "."  # change if you want to move the Excel somewhere else

# === Load CSV Data (RAW ONLY) ===
df = pd.read_csv("Final_Data_Set_With_Nitorgen_Suzzy.csv")
df = df.drop(columns=["Sample","Name"], errors="ignore")
target_col = "N%"
feature_df = df.drop(columns=[target_col])
wavelength_names = feature_df.columns.to_numpy()
X_raw = feature_df.values
y = df[target_col].values

# === Helpers ===
def compute_vip(pls: PLSRegression) -> np.ndarray:
    """
    VIP for PLS1. Uses fitted attributes of pls (after fit).
    """
    T = pls.x_scores_                   # (n_samples, A)
    W = pls.x_weights_                  # (p, A)
    Q = pls.y_loadings_.ravel()         # (A,)
    p, A = W.shape

    SS = np.sum((T**2) * (Q**2), axis=0)     # (A,)
    total_SS = np.sum(SS) + 1e-12

    vip = np.zeros(p)
    for j in range(p):
        vip[j] = np.sqrt(p * np.sum((W[j, :]**2) * SS) / total_SS)
    return vip

def save_actual_vs_pred_plot(ytr, ytr_hat, ycv_hat, yte, yte_hat, title, outfile):
    plt.figure(figsize=(6,6))
    plt.scatter(ytr, ytr_hat, alpha=0.6, label=f"Train (R²={r2_score(ytr, ytr_hat):.3f})")
    plt.scatter(ytr, ycv_hat, alpha=0.6, label=f"Val (R²={r2_score(ytr, ycv_hat):.3f})")
    plt.scatter(yte, yte_hat, alpha=0.9, label=f"Test (R²={r2_score(yte, yte_hat):.3f})")
    all_y = np.concatenate([ytr, yte])
    lo, hi = np.min(all_y), np.max(all_y)
    plt.plot([lo, hi], [lo, hi], '--', lw=1)
    plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(outfile, dpi=200); plt.close()

# === Config (Raw only) ===
repeats = 10
# n_components search upper bound (trimmed per split to min(n-1, p))
max_ncomp = 20
# try these top-k wavelengths via VIP (None means "use all")
k_grid = [50, 75, 100, 150, 200, 300, None]

summary_data = {}

with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    method_name = "Raw"
    print(f"🔁 Running: {method_name}")
    results = []

    X_processed = X_raw  # RAW ONLY

    for seed in range(repeats):
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_processed, y, test_size=0.1, random_state=seed
        )

        # --- Stage 1: Fit a base PLS on TRAIN to get VIP ranking (no leakage) ---
        # choose a safe base component count
        base_max = int(min(15, X_train.shape[1]-1, max(2, np.linalg.matrix_rank(X_train))))
        base_max = max(2, base_max)
        pls_base = PLSRegression(n_components=base_max)
        pls_base.fit(X_train, y_train)
        vip = compute_vip(pls_base)
        vip_order = np.argsort(vip)[::-1]  # desc

        # --- Stage 2: grid search over (top-k wavelengths) x (n_components) with CV on TRAIN ---
        best = {"cv_mse": np.inf}
        for k in k_grid:
            if (k is None) or (k >= X_train.shape[1]):
                sel_idx = np.arange(X_train.shape[1])
                k_used = X_train.shape[1]
            else:
                sel_idx = vip_order[:k]
                k_used = int(k)

            Xtr_sel = X_train[:, sel_idx]

            # cap n_components to feasible range
            max_allowed = int(min(max_ncomp, Xtr_sel.shape[0]-1, Xtr_sel.shape[1]))
            if max_allowed < 1:
                continue

            grid = GridSearchCV(
                PLSRegression(),
                {'n_components': list(range(1, max_allowed+1))},
                cv=5,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                return_train_score=True
            )
            grid.fit(Xtr_sel, y_train)
            cv_mse = -grid.best_score_

            if cv_mse < best["cv_mse"]:
                best.update({
                    "cv_mse": cv_mse,
                    "k": k_used,
                    "n_components": grid.best_params_["n_components"],
                    "sel_idx": sel_idx
                })

        # --- Stage 3: refit best model on selected wavelengths; evaluate & plot ---
        sel_idx = best["sel_idx"]
        Xtr_sel = X_train[:, sel_idx]
        Xte_sel = X_test[:, sel_idx]

        best_model = PLSRegression(n_components=best["n_components"])
        best_model.fit(Xtr_sel, y_train)

        ytr_hat = best_model.predict(Xtr_sel).ravel()
        ycv_hat = cross_val_predict(best_model, Xtr_sel, y_train, cv=5).ravel()
        yte_hat = best_model.predict(Xte_sel).ravel()

        train_r2 = r2_score(y_train, ytr_hat)
        train_rmse = np.sqrt(mean_squared_error(y_train, ytr_hat))
        train_mae = mean_absolute_error(y_train, ytr_hat)

        val_r2 = r2_score(y_train, ycv_hat)
        val_rmse = np.sqrt(mean_squared_error(y_train, ycv_hat))
        val_mae = mean_absolute_error(y_train, ycv_hat)

        test_r2 = r2_score(y_test, yte_hat)
        test_rmse = np.sqrt(mean_squared_error(y_test, yte_hat))
        test_mae = mean_absolute_error(y_test, yte_hat)
        test_rpd = np.std(y_test) / np.sqrt(mean_squared_error(y_test, yte_hat))

        title = f"{method_name} | Repeat {seed+1} | k={best['k']} | n={best['n_components']}"
        plot_path = os.path.join(plots_dir, f"{method_name}_repeat{seed+1}_k{best['k']}_n{best['n_components']}.png")
        save_actual_vs_pred_plot(y_train, ytr_hat, ycv_hat, y_test, yte_hat, title, plot_path)

        # save selected wavelengths (names)
        pd.Series(wavelength_names[sel_idx], name="wavelength").to_csv(
            os.path.join(waves_dir, f"{method_name}_repeat{seed+1}_selected_wavelengths.csv"),
            index=False
        )

        results.append({
            "Repeat": seed + 1,
            "Train R²": train_r2,
            "Train RMSE": train_rmse,
            "Train MAE": train_mae,
            "Val R²": val_r2,
            "Val RMSE": val_rmse,
            "Val MAE": val_mae,
            "Test R²": test_r2,
            "Test RMSE": test_rmse,
            "Test MAE": test_mae,
            "Test RPD": test_rpd,
            "Best n_components": best["n_components"],
            "Best k (wavelengths)": best["k"],
            "Plot Path": plot_path
        })

    # Write per-repeat + Mean/Std
    df_result = pd.DataFrame(results)
    df_result.loc["Mean"] = df_result.drop(columns=["Repeat","Best n_components","Best k (wavelengths)","Plot Path"]).mean(numeric_only=True)
    df_result.loc["Std"]  = df_result.drop(columns=["Repeat","Best n_components","Best k (wavelengths)","Plot Path"]).std(numeric_only=True)
    df_result.to_excel(writer, sheet_name=method_name, index=False)

    # Compact summary sheet (Mean ± Std for key metrics)
    cols = ["Train R²","Val R²","Test R²","Train RMSE","Val RMSE","Test RMSE","Test RPD"]
    summary = {c: f"{df_result.loc['Mean', c]:.4f} ± {df_result.loc['Std', c]:.4f}" for c in cols}
    pd.DataFrame([summary], index=[method_name]).to_excel(writer, sheet_name="Summary")

print(f"✅ All results saved to {output_filename}")
try:
    shutil.move(output_filename, f"{destination_path}/{output_filename}")
    print(f"✅ File moved to: {destination_path}/{output_filename}")
except Exception as e:
    print(f"ℹ️ Skipped moving Excel (set destination_path). Reason: {e}")


🔁 Running: Raw
✅ All results saved to PLSR_Full_Preprocessing_Comparison2.xlsx
✅ File moved to: ./PLSR_Full_Preprocessing_Comparison2.xlsx


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
INPUT_CSV = "Final_Data_Set_With_Nitorgen_Suzzy.csv"
TARGET_COL = "N%"
REPEATS = 10
ALPHAS = [0.0001, 0.001, 0.01, 0.1, 1, 10]
OUTPUT_XLSX = "LassoRegression_Full_Preprocessing_Comparison.xlsx"
TOP_K_OVERALL = 10  # how many wavelengths to show in the summary top list

PLOTS_DIR = "plots"
SEL_DIR = "selected_wavelengths"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(SEL_DIR, exist_ok=True)

# =========================
# Load & clean data (RAW)
# =========================
df = pd.read_csv(INPUT_CSV)
df = df.drop(columns=["Sample", "Name"], errors="ignore")

# Coerce features to numeric and handle NaNs safely
feature_df = df.drop(columns=[TARGET_COL]).apply(pd.to_numeric, errors="coerce")
feature_df = feature_df.dropna(axis=1, how="all")
feature_df = feature_df.fillna(method="ffill").fillna(method="bfill")
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

wavelength_names = feature_df.columns.to_numpy()
X_raw = feature_df.to_numpy(dtype=np.float64)
y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)

assert np.isfinite(X_raw).all(), "X has non-finite values after cleaning."
assert np.isfinite(y).all(), "y has non-finite values after cleaning."

# =========================
# Plot helpers
# =========================
def save_actual_vs_pred_plot(y_train, y_train_pred, y_val_pred, y_test, y_test_pred,
                             title: str, outfile: str):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_train, y_train_pred, alpha=0.6,
                label=f"Train (R²={r2_score(y_train, y_train_pred):.3f})")
    plt.scatter(y_train, y_val_pred, alpha=0.6,
                label=f"Val (CV) (R²={r2_score(y_train, y_val_pred):.3f})")
    plt.scatter(y_test, y_test_pred, alpha=0.9,
                label=f"Test (R²={r2_score(y_test, y_test_pred):.3f})")
    lo = min(np.min(y_train), np.min(y_test))
    hi = max(np.max(y_train), np.max(y_test))
    plt.plot([lo, hi], [lo, hi], '--', lw=1)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

def save_single_split_plot(y_true, y_pred, title, outfile):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.9)
    lo, hi = np.min(y_true), np.max(y_true)
    plt.plot([lo, hi], [lo, hi], '--', lw=1)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{title} (R²={r2_score(y_true, y_pred):.3f})")
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

# =========================
# Run repeats
# =========================
results = []
all_importances = []  # per-repeat importance tables

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    print("🟠 Running: Raw (Lasso → OLS)")

    for seed in range(REPEATS):
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_raw, y, test_size=0.1, random_state=seed
        )

        # Pipeline: Standardize -> Lasso
        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("lasso", Lasso(max_iter=10000))
        ])

        grid = GridSearchCV(
            pipe,
            {"lasso__alpha": ALPHAS},
            cv=5,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            return_train_score=True
        )
        grid.fit(X_train, y_train)
        best_pipe = grid.best_estimator_
        best_alpha = grid.best_params_["lasso__alpha"]

        scaler: StandardScaler = best_pipe.named_steps["scaler"]
        lasso: Lasso = best_pipe.named_steps["lasso"]

        coef_std = lasso.coef_.copy()  # coefficients in standardized space
        selected_idx = np.flatnonzero(coef_std != 0)

        if selected_idx.size == 0:
            # Fallback: if nothing selected, predict mean
            y_train_pred = np.full_like(y_train, y_train.mean(), dtype=float)
            y_val_pred = cross_val_predict(
                LinearRegression(), np.ones((len(y_train), 1)), y_train, cv=5
            )
            y_test_pred = np.full_like(y_test, y_train.mean(), dtype=float)
            selected_count = 0
        else:
            # Convert coefficients back to original feature units:
            # coef_orig = coef_std / scale
            scale = scaler.scale_
            coef_orig = np.zeros_like(coef_std)
            coef_orig[selected_idx] = coef_std[selected_idx] / scale[selected_idx]

            # Importance table for this repeat
            imp_df = pd.DataFrame({
                "wavelength": wavelength_names,
                "coef_std": coef_std,
                "abs_coef_std": np.abs(coef_std),
                "coef_orig": coef_orig,
                "abs_coef_orig": np.abs(coef_orig),
                "selected": (coef_std != 0).astype(int),
                "repeat": seed + 1,
                "alpha": best_alpha
            })
            all_importances.append(imp_df)

            # Refit OLS on selected features
            Xtr_sel = X_train[:, selected_idx]
            Xte_sel = X_test[:, selected_idx]
            ols = LinearRegression().fit(Xtr_sel, y_train)

            y_train_pred = ols.predict(Xtr_sel)
            y_val_pred = cross_val_predict(ols, Xtr_sel, y_train, cv=5)
            y_test_pred = ols.predict(Xte_sel)

            # Save per-repeat selected wavelengths
            sel_table = imp_df.loc[imp_df["selected"] == 1].sort_values(
                "abs_coef_std", ascending=False
            )
            sel_table.to_csv(
                os.path.join(SEL_DIR, f"Raw_repeat{seed+1}_alpha{best_alpha}_selected.csv"),
                index=False
            )
            selected_count = int(selected_idx.size)

        # Metrics
        train_r2 = r2_score(y_train, y_train_pred)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        train_mae = mean_absolute_error(y_train, y_train_pred)

        val_r2 = r2_score(y_train, y_val_pred)
        val_rmse = np.sqrt(mean_squared_error(y_train, y_val_pred))
        val_mae = mean_absolute_error(y_train, y_val_pred)

        test_r2 = r2_score(y_test, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rpd = np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred))

        # Plots (combined + separate)
        title = f"Raw | Repeat {seed+1} | alpha={best_alpha} | selected={selected_count}"
        base = f"Raw_repeat{seed+1}_alpha{best_alpha}"

        save_actual_vs_pred_plot(
            y_train, y_train_pred, y_val_pred, y_test, y_test_pred,
            title=title,
            outfile=os.path.join(PLOTS_DIR, f"{base}_ALL.png")
        )
        save_single_split_plot(
            y_train, y_train_pred,
            f"Train | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_TRAIN.png")
        )
        save_single_split_plot(
            y_train, y_val_pred,
            f"Val (CV) | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_VAL.png")
        )
        save_single_split_plot(
            y_test, y_test_pred,
            f"Test | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_TEST.png")
        )

        # Row
        results.append({
            "Repeat": seed + 1,
            "Train R²": train_r2,
            "Train RMSE": train_rmse,
            "Train MAE": train_mae,
            "Val R²": val_r2,
            "Val RMSE": val_rmse,
            "Val MAE": val_mae,
            "Test R²": test_r2,
            "Test RMSE": test_rmse,
            "Test MAE": test_mae,
            "Test RPD": test_rpd,
            "Best Params": str({"alpha": best_alpha}),
            "Selected Features": selected_count
        })

    # Per-repeat sheet + Mean/Std
    df_result = pd.DataFrame(results)
    df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean(numeric_only=True)
    df_result.loc["Std"]  = df_result.drop(columns=["Repeat", "Best Params"]).std(numeric_only=True)
    df_result.to_excel(writer, sheet_name="Raw", index=False)

    # Summary sheet (Mean ± Std strings)
    summary_data = {
        "Raw": df_result.loc["Mean"].astype(str) + " ± " + df_result.loc["Std"].round(4).astype(str)
    }
    pd.DataFrame(summary_data).T.to_excel(writer, sheet_name="Summary")

    # Cross-repeat importance aggregation
    if len(all_importances):
        imp_all = pd.concat(all_importances, ignore_index=True)
        agg = imp_all.groupby("wavelength").agg(
            times_selected=("selected", "sum"),
            mean_abs_coef_std=("abs_coef_std", "mean"),
            median_abs_coef_std=("abs_coef_std", "median"),
            mean_abs_coef_orig=("abs_coef_orig", "mean"),
            median_abs_coef_orig=("abs_coef_orig", "median")
        ).reset_index()

        # Rank by standardized magnitude (change to *_orig if you prefer)
        agg = agg.sort_values(["mean_abs_coef_std", "times_selected"], ascending=[False, False])

        agg.to_excel(writer, sheet_name="Importance_All", index=False)
        agg.head(TOP_K_OVERALL).to_excel(writer, sheet_name="Top_Importance_Raw", index=False)

print(f"✅ Done. Results in {OUTPUT_XLSX}, plots in '{PLOTS_DIR}/', selections in '{SEL_DIR}/'.")


🟠 Running: Raw (Lasso → OLS)
✅ Done. Results in LassoRegression_Full_Preprocessing_Comparison.xlsx, plots in 'plots/', selections in 'selected_wavelengths/'.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
INPUT_CSV = "Final_Data_Set_With_Nitorgen_Suzzy.csv"
TARGET_COL = "N%"
REPEATS = 10
ALPHAS = [0.0001, 0.001, 0.01, 0.1, 1, 10]
OUTPUT_XLSX = "LassoRegression_FixedBands.xlsx"

# parsing/matching tolerance in nm for mapping λ-wavelengths to column names
TOL = 1.0

PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# ======== Your fixed top wavelengths & importance table (as provided) ========
top_table = pd.DataFrame(
    {
        "wavelength": [
            "λ471.98", "λ543.66", "λ967.74", "λ507.82", "λ412.25",
            "λ940.86", "λ483.93", "λ961.77", "λ973.71", "λ902.04"
        ],
        "times_selected": [9, 10, 9, 9, 9, 9, 9, 9, 8, 9],
        "mean_abs_coef_std": [1.498950168, 1.249026814, 1.02061234, 0.982555147, 0.875559475, 0.862413616, 0.818086621, 0.647830084, 0.466988674, 0.439592689],
        "median_abs_coef_std": [1.612927715, 1.229439897, 1.038444811, 1.073177585, 1.000360844, 0.999633377, 0.965911644, 0.478248661, 0.520658559, 0.408345927],
        "mean_abs_coef_orig": [147.8927806, 18.89878349, 55.53757123, 48.40899476, 12.69596965, 52.10522846, 124.311464, 36.38673987, 23.58712621, 24.95139845],
        "median_abs_coef_orig": [157.8958177, 18.28857045, 56.28844147, 51.73448644, 14.71981831, 59.73762795, 149.2521072, 27.13149379, 26.82248964, 22.55610961],
    }
)

def parse_lambda(s: str) -> float:
    """Turn 'λ471.98' or '471.98' into float 471.98."""
    s = str(s).strip()
    s = s.replace("λ", "").replace("Lambda", "").strip()
    return float(s)

fixed_wavelengths_nm = [parse_lambda(w) for w in top_table["wavelength"].tolist()]

# =========================
# Load & clean data (RAW)
# =========================
df = pd.read_csv(INPUT_CSV)
df = df.drop(columns=["Sample", "Name"], errors="ignore")

# Coerce features to numeric and handle NaNs safely
feature_df = df.drop(columns=[TARGET_COL]).apply(pd.to_numeric, errors="coerce")
feature_df = feature_df.dropna(axis=1, how="all")
feature_df = feature_df.fillna(method="ffill").fillna(method="bfill")
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

# Prepare arrays
X_all = feature_df.to_numpy(dtype=np.float64)
y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
assert np.isfinite(X_all).all() and np.isfinite(y).all()

# Map column names to numeric wavelengths (best effort)
def col_to_nm(col):
    try:
        return float(str(col).replace("λ", "").strip())
    except Exception:
        return np.nan

col_names = feature_df.columns.to_numpy()
col_nms = np.array([col_to_nm(c) for c in col_names], dtype=np.float64)

# Build mapping from requested nm -> dataset column (exact or nearest within TOL)
resolved = []
sel_indices = []
for w in fixed_wavelengths_nm:
    # try exact match among numeric-convertible columns
    exact_matches = np.where(np.isfinite(col_nms) & (np.isclose(col_nms, w, rtol=0.0, atol=1e-6)))[0]
    if exact_matches.size > 0:
        j = int(exact_matches[0])
        resolved.append((w, col_names[j], col_nms[j], 0.0, "exact"))
        sel_indices.append(j)
    else:
        # nearest neighbor within tolerance
        diffs = np.abs(col_nms - w)
        if np.isfinite(diffs).any():
            j = int(np.nanargmin(diffs))
            gap = float(diffs[j])
            if gap <= TOL:
                resolved.append((w, col_names[j], col_nms[j], gap, "nearest"))
                sel_indices.append(j)
            else:
                resolved.append((w, None, np.nan, gap, "unmatched"))
        else:
            resolved.append((w, None, np.nan, np.nan, "unmatched"))

resolve_df = pd.DataFrame(resolved, columns=["requested_nm", "matched_column", "matched_nm", "delta_nm", "mode"])

# Use only matched indices (unique, preserve order)
sel_indices = [i for i in sel_indices if i is not None]
# de-duplicate while preserving order
seen = set()
sel_indices_unique = []
for i in sel_indices:
    if i not in seen:
        sel_indices_unique.append(i)
        seen.add(i)

if len(sel_indices_unique) == 0:
    raise ValueError("None of the requested wavelengths could be matched to dataset columns within tolerance. "
                     "Increase TOL or check column labels.")

X_fixed = X_all[:, sel_indices_unique]
fixed_cols = col_names[sel_indices_unique]

# =========================
# Plot helpers
# =========================
def save_actual_vs_pred_plot(y_train, y_train_pred, y_val_pred, y_test, y_test_pred,
                             title: str, outfile: str):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_train, y_train_pred, alpha=0.6,
                label=f"Train (R²={r2_score(y_train, y_train_pred):.3f})")
    plt.scatter(y_train, y_val_pred, alpha=0.6,
                label=f"Val (CV) (R²={r2_score(y_train, y_val_pred):.3f})")
    plt.scatter(y_test, y_test_pred, alpha=0.9,
                label=f"Test (R²={r2_score(y_test, y_test_pred):.3f})")
    lo = min(np.min(y_train), np.min(y_test))
    hi = max(np.max(y_train), np.max(y_test))
    plt.plot([lo, hi], [lo, hi], '--', lw=1)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

def save_single_split_plot(y_true, y_pred, title, outfile):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.9)
    lo, hi = np.min(y_true), np.max(y_true)
    plt.plot([lo, hi], [lo, hi], '--', lw=1)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{title} (R²={r2_score(y_true, y_pred):.3f})")
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

# =========================
# Train with fixed wavelengths
# =========================
results = []

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    # save your provided importance table & the resolved mapping
    top_table.to_excel(writer, sheet_name="Provided_Top10", index=False)
    resolve_df.to_excel(writer, sheet_name="Resolved_Columns", index=False)

    print(f"🔧 Using {len(sel_indices_unique)} matched columns from your Top10 list.")

    for seed in range(REPEATS):
        X_train, X_test, y_train, y_test = train_test_split(
            X_fixed, y, test_size=0.1, random_state=seed
        )

        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("lasso", Lasso(max_iter=10000))
        ])

        grid = GridSearchCV(
            pipe,
            {"lasso__alpha": ALPHAS},
            cv=5,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            return_train_score=True
        )
        grid.fit(X_train, y_train)
        best_pipe = grid.best_estimator_
        best_alpha = grid.best_params_["lasso__alpha"]

        scaler: StandardScaler = best_pipe.named_steps["scaler"]
        lasso: Lasso = best_pipe.named_steps["lasso"]

        coef_std = lasso.coef_.copy()
        selected_idx_local = np.flatnonzero(coef_std != 0)

        if selected_idx_local.size == 0:
            # fallback
            y_train_pred = np.full_like(y_train, y_train.mean(), dtype=float)
            y_val_pred = cross_val_predict(
                LinearRegression(), np.ones((len(y_train), 1)), y_train, cv=5
            )
            y_test_pred = np.full_like(y_test, y_train.mean(), dtype=float)
            selected_count = 0
        else:
            Xtr_sel = X_train[:, selected_idx_local]
            Xte_sel = X_test[:, selected_idx_local]
            ols = LinearRegression().fit(Xtr_sel, y_train)
            y_train_pred = ols.predict(Xtr_sel)
            y_val_pred = cross_val_predict(ols, Xtr_sel, y_train, cv=5)
            y_test_pred = ols.predict(Xte_sel)
            selected_count = int(selected_idx_local.size)

        # metrics
        train_r2 = r2_score(y_train, y_train_pred)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        train_mae = mean_absolute_error(y_train, y_train_pred)

        val_r2 = r2_score(y_train, y_val_pred)
        val_rmse = np.sqrt(mean_squared_error(y_train, y_val_pred))
        val_mae = mean_absolute_error(y_train, y_val_pred)

        test_r2 = r2_score(y_test, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rpd = np.std(y_test) / np.sqrt(mean_squared_error(y_test, y_test_pred))

        # plots
        base = f"FixedBands_repeat{seed+1}_alpha{best_alpha}_sel{selected_count}"
        save_actual_vs_pred_plot(
            y_train, y_train_pred, y_val_pred, y_test, y_test_pred,
            title=f"Fixed Top Bands | Repeat {seed+1} | alpha={best_alpha} | sel={selected_count}",
            outfile=os.path.join(PLOTS_DIR, f"{base}_ALL.png")
        )
        save_single_split_plot(
            y_train, y_train_pred,
            f"Train | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_TRAIN.png")
        )
        save_single_split_plot(
            y_train, y_val_pred,
            f"Val (CV) | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_VAL.png")
        )
        save_single_split_plot(
            y_test, y_test_pred,
            f"Test | Repeat {seed+1} | alpha={best_alpha}",
            os.path.join(PLOTS_DIR, f"{base}_TEST.png")
        )

        results.append({
            "Repeat": seed + 1,
            "Train R²": train_r2,
            "Train RMSE": train_rmse,
            "Train MAE": train_mae,
            "Val R²": val_r2,
            "Val RMSE": val_rmse,
            "Val MAE": val_mae,
            "Test R²": test_r2,
            "Test RMSE": test_rmse,
            "Test MAE": test_mae,
            "Test RPD": test_rpd,
            "Best Params": str({"alpha": best_alpha}),
            "Selected (from Top10)": selected_count
        })

    # write results
    df_result = pd.DataFrame(results)
    df_result.loc["Mean"] = df_result.drop(columns=["Repeat", "Best Params"]).mean(numeric_only=True)
    df_result.loc["Std"]  = df_result.drop(columns=["Repeat", "Best Params"]).std(numeric_only=True)
    df_result.to_excel(writer, sheet_name="FixedBands_Results", index=False)

print(f"✅ Done. Results in {OUTPUT_XLSX}. Plots saved in '{PLOTS_DIR}/'.")


🔧 Using 10 matched columns from your Top10 list.
✅ Done. Results in LassoRegression_FixedBands.xlsx. Plots saved in 'plots/'.
